# SPARQL query tests

Run the cells in order or use **Run All**. The Turtle graph is loaded once, without a reasoner. Queries are read from `rdf/queries.sparql`; rerun the cells after modifying that file. The first seven checks should return zero rows. The remaining searches may return no results for this sample.


If RDFLib or pandas is not available in the kernel, run `%pip install rdflib pandas` in a cell, then restart the kernel if necessary.


In [16]:
from pathlib import Path
import re
import pandas as pd
from IPython.display import display
from rdflib import Graph

# Works from transformation/, Group07, or the repository root.
candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "HandsOn" / "Group07",
]
group_dir = next(
    (path for path in candidates if (path / "rdf" / "queries.sparql").is_file()),
    None,
)
if group_dir is None:
    raise FileNotFoundError("Open the notebook from transformation/, Group07, or the repository root.")

graph = Graph()
graph.parse(group_dir / "rdf" / "knowledge-graph.ttl", format="turtle")
print(f"Graph loaded: {len(graph):,} triples")

source = (group_dir / "rdf" / "queries.sparql").read_text(encoding="utf-8-sig")
# Split this file into blocks containing one SELECT query each.
parts = re.split(r"(?m)^SELECT\b", source)
prefixes = "\n".join(
    line for line in parts[0].splitlines()
    if line.strip().upper().startswith("PREFIX ")
)
queries = [prefixes + "\nSELECT" + part for part in parts[1:]]
if len(queries) != 12:
    raise ValueError(f"Expected 12 queries, found {len(queries)}. Update the cells if the list changes.")
print(f"{len(queries)} queries available")


def run_query(index, expected_empty=False):
    result = graph.query(queries[index - 1])
    rows = list(result)
    table = pd.DataFrame(
        [[None if value is None else str(value) for value in row] for row in rows],
        columns=[str(variable) for variable in result.vars],
    )
    if expected_empty:
        print("OK: no anomaly detected." if not rows else f"Check: {len(rows)} result(s).")
    else:
        print(f"{len(rows)} result(s).")
    display(table.head(50))
    if len(table) > 50:
        print("Display limited to the first 50 rows; all results remain available in the DataFrame.")
    return table


Graph loaded: 16,552 triples
12 queries available


## 1. TrainStop without a station

Expected result: zero rows.


In [2]:
results_1 = run_query(1, expected_empty=True)


OK: no anomaly detected.


,stop


## 2. TrainStop with more than one station

Expected result: zero rows.


In [3]:
results_2 = run_query(2, expected_empty=True)


OK: no anomaly detected.


,stop,n


## 3. Station without exactly one EVA number

Expected result: zero rows.


In [4]:
results_3 = run_query(3, expected_empty=True)


OK: no anomaly detected.


,st,n


## 4. Several Station individuals sharing one EVA number

Expected result: zero rows.


In [5]:
results_4 = run_query(4, expected_empty=True)


OK: no anomaly detected.


,eva,n


## 5. Broken Station -> City -> State chain

Expected result: zero rows.


In [6]:
results_5 = run_query(5, expected_empty=True)


OK: no anomaly detected.


,st,city


## 6. City located in more than one State

Expected result: zero rows.


In [7]:
results_6 = run_query(6, expected_empty=True)


OK: no anomaly detected.


,city,n


## 7. Domain/range violations (run WITHOUT a reasoner)

Expected result: zero rows.


In [8]:
results_7 = run_query(7, expected_empty=True)


OK: no anomaly detected.


,s,o


## 8. Instance counts per class

An empty result may depend on the sample data.


In [9]:
results_8 = run_query(8, expected_empty=False)


4 result(s).


,class,n
0,https://example.org/db-delays/ontology/#TrainStop,1000
1,https://example.org/db-delays/ontology/#Station,251
2,https://example.org/db-delays/ontology/#City,167
3,https://example.org/db-delays/ontology/#State,16


## 9. Stops at a given station (adjust EVA), with delays

An empty result may depend on the sample data.


In [17]:
results_9 = run_query(9, expected_empty=False)


6 result(s).


,stop,line,plannedArrival,arrivalDelayMinutes
0,https://example.org/db-delays/resource/TrainSt...,7,2024-07-08T00:13:00,0
1,https://example.org/db-delays/resource/TrainSt...,7,2024-07-08T00:16:00,0
2,https://example.org/db-delays/resource/TrainSt...,7,2024-07-08T00:33:00,0
3,https://example.org/db-delays/resource/TrainSt...,7,2024-07-08T00:36:00,0
4,https://example.org/db-delays/resource/TrainSt...,7,2024-07-08T00:53:00,0
5,https://example.org/db-delays/resource/TrainSt...,7,2024-07-08T00:56:00,0


## 10. Stops whose path mentions a given previous station

An empty result may depend on the sample data.


In [11]:
results_10 = run_query(10, expected_empty=False)


18 result(s).


,stop,path
0,https://example.org/db-delays/resource/TrainSt...,Köln Hbf|Köln Messe/Deutz|Porz(Rhein)|Troisdor...
1,https://example.org/db-delays/resource/TrainSt...,Köln Hbf|Köln Messe/Deutz|Porz(Rhein)|Troisdor...
2,https://example.org/db-delays/resource/TrainSt...,Köln Hbf|Köln Messe/Deutz|Porz(Rhein)|Troisdor...
3,https://example.org/db-delays/resource/TrainSt...,Köln/Bonn Flughafen|Köln Messe/Deutz|Köln Hbf|...
4,https://example.org/db-delays/resource/TrainSt...,Köln/Bonn Flughafen|Köln Messe/Deutz|Köln Hbf|...
5,https://example.org/db-delays/resource/TrainSt...,Köln/Bonn Flughafen|Köln Messe/Deutz|Köln Hbf|...
6,https://example.org/db-delays/resource/TrainSt...,Köln/Bonn Flughafen|Köln Messe/Deutz|Köln Hbf|...
7,https://example.org/db-delays/resource/TrainSt...,Hennef(Sieg)|Siegburg/Bonn|Troisdorf|Spich|Por...
8,https://example.org/db-delays/resource/TrainSt...,Aachen Hbf|Aachen-Rothe Erde|Stolberg(Rheinl)H...
9,https://example.org/db-delays/resource/TrainSt...,Köln/Bonn Flughafen|Köln Messe/Deutz|Köln Hbf|...


## 11. Stations in a bounding box (around Berlin)

An empty result may depend on the sample data.


In [12]:
results_11 = run_query(11, expected_empty=False)


55 result(s).


,name,lat,long
0,Berlin-Karlshorst,52.4797942,13.5279465
1,Blankenfelde (Kr Teltow-Fläming),52.337213,13.415982
2,Berlin Ostbahnhof,52.5104881,13.4346807
3,Berlin-Charlottenburg,52.5042126,13.3023298
4,Berlin-Spandau,52.5346481,13.1968975
5,Berlin-Wannsee,52.4209723,13.1795275
6,Berlin Zoologischer Garten,52.5073407,13.3324447
7,Berlin-Karow,52.614884,13.469204
8,Berlin Gesundbrunnen,52.548963,13.388513
9,Alexanderplatz,52.521481,13.410961


Display limited to the first 50 rows; all results remain available in the DataFrame.


## 12. Average arrival delay per state

An empty result may depend on the sample data.


In [13]:
results_12 = run_query(12, expected_empty=False)


16 result(s).


,stateName,avgDelay,stops
0,Niedersachsen,8.324324324324324324324324324,37
1,Thüringen,3.5,4
2,Bremen,3.0625,32
3,Sachsen,2.176470588235294117647058824,17
4,Nordrhein-Westfalen,1.761904761904761904761904762,84
5,Bayern,1.6,55
6,Baden-Württemberg,1.064935064935064935064935065,77
7,Mecklenburg-Vorpommern,1,5
8,Sachsen-Anhalt,1,6
9,Hessen,0.9767441860465116279069767442,43
